# 17.2 知识编辑 (Knowledge Editing)

知识编辑使模型在不需要完全重新训练的情况下更新特定知识。

本节涵盖：模型编辑、知识注入、知识更新

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class KnowledgeEditor(nn.Module):
    def __init__(self, d=64, n_layers=2):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(d, d) for _ in range(n_layers)])
        self.hyper_net = nn.Linear(d * 2, d * d)

    def edit_layer(self, layer_idx, subject, target):
        W = self.layers[layer_idx].weight
        u = subject / (subject.norm() + 1e-8)
        delta = target - (W @ subject)
        W_new = W + torch.outer(delta.squeeze(), u.squeeze()) / (u.norm() ** 2 + 1e-8)
        self.layers[layer_idx].weight = nn.Parameter(W_new)

    def forward(self, x):
        for layer in self.layers:
            x = F.relu(layer(x))
        return x

editor = KnowledgeEditor()
subject = torch.randn(1, 64)
original_output = editor(subject)

target_direction = torch.randn(1, 64)
editor.edit_layer(0, subject, target_direction)
edited_output = editor(subject)

print('=== Knowledge Editing ===')
print(f'\nMethods:')
print(f'  ROME: Rank-one model editing (modify specific MLP weights)')
print(f'  MEMIT: Mass-editing via constraint optimization')
print(f'  SERAC: Counterfactual model for targeted edits')
print(f'\nDemo: Rank-one edit on layer 0')
print(f'  Original output norm: {original_output.norm():.4f}')
print(f'  Edited output norm: {edited_output.norm():.4f}')
print(f'\nKey: Knowledge editing updates specific facts without full retraining.')
print(f'Useful for correcting errors, updating facts, removing harmful knowledge.')